In [52]:
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score



In [53]:
bank = pd.read_csv('/home/aiden/projects/ML/banking/dataset/bank-additional-full.csv', sep=';')

In [54]:
bank["y"]=bank["y"].map({"yes":1, "no":0}).astype("int")

In [55]:
def categorize_pdays(col):
    if col==999:
        return "never-contacted"
    elif col<=7:
        return "recent"
    elif col<=14:
        return "mid"
    elif col<999:
        return "late"
bank["pdays_cat"]=bank["pdays"].apply(categorize_pdays).astype("category")
bank["pdays_cat"].value_counts()
bank.drop("pdays", axis=1, inplace=True)
bank.drop("duration", axis=1, inplace=True)

In [56]:
split_idx = int(len(bank) * 0.80)

train = bank.iloc[:split_idx].copy()
test = bank.iloc[split_idx:].copy()

Y_train = train['y'].copy()
X_train = train.drop(columns=['y'])

X_test = test.drop(columns=['y'])
Y_test = test['y'].copy()

In [57]:
bank.info()

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             41188 non-null  int64   
 1   job             41188 non-null  str     
 2   marital         41188 non-null  str     
 3   education       41188 non-null  str     
 4   default         41188 non-null  str     
 5   housing         41188 non-null  str     
 6   loan            41188 non-null  str     
 7   contact         41188 non-null  str     
 8   month           41188 non-null  str     
 9   day_of_week     41188 non-null  str     
 10  campaign        41188 non-null  int64   
 11  previous        41188 non-null  int64   
 12  poutcome        41188 non-null  str     
 13  emp.var.rate    41188 non-null  float64 
 14  cons.price.idx  41188 non-null  float64 
 15  cons.conf.idx   41188 non-null  float64 
 16  euribor3m       41188 non-null  float64 
 17  nr.employed     41188 n

In [58]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# 1. Feature lists (Added euribor3m and cons.conf.idx)
nums = ["age", "previous", "euribor3m", "cons.price.idx", "cons.conf.idx"]
cats = ["job", "marital", "education", "housing", "loan", "contact","day_of_week", "poutcome", "pdays_cat"]

log_pipeline = make_pipeline(
    FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
    StandardScaler()
)

# 3. Preprocessor
LogPrep = ColumnTransformer(
    transformers=[
        ('log_camp', log_pipeline, ["campaign"]),
        ('nums', StandardScaler(), nums),
        ('cats', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cats)
    ], 
    remainder='drop'
)

logreg = Pipeline(steps=[
    ('preprocessor', LogPrep),
    ('classifier', LogisticRegression(                    
        max_iter=1000,
        class_weight='balanced',
        solver='lbfgs',
        random_state=42
    ))
])

logreg.fit(X_train, Y_train)

y_prob = logreg.predict_proba(X_test)[:, 1]


adjusted_threshold = 0.65
y_pred_adjusted = (y_prob >= adjusted_threshold).astype(int)

print(classification_report(Y_test, y_pred_adjusted, target_names=['rejected (0)', 'accepted (1)']))



              precision    recall  f1-score   support

rejected (0)       0.72      0.60      0.65      5698
accepted (1)       0.35      0.47      0.40      2540

    accuracy                           0.56      8238
   macro avg       0.53      0.54      0.53      8238
weighted avg       0.60      0.56      0.58      8238



/home/aiden/projects/ML/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [59]:
from sklearn.ensemble import RandomForestClassifier

rf_nums=["age", "campaign", "previous","emp.var.rate", "cons.price.idx", "nr.employed", "euribor3m", "cons.conf.idx"]
rf_cats=["job", "marital", "education", "default", "housing", "loan", "contact", "day_of_week", "poutcome", "pdays_cat"]

rf_prep = ColumnTransformer(
    transformers=[
        ('nums', StandardScaler(), rf_nums),
        ('cats', OneHotEncoder(handle_unknown='ignore'), rf_cats)
    ], remainder='drop'
)

forest=Pipeline(steps=[('preprocessor', rf_prep),
                          ('classifier', RandomForestClassifier(
                                n_estimators=300,
                                class_weight='balanced',
                                 random_state=42,
                                 n_jobs=-1,
                                 max_depth=10
                                 ))
                            ])

forest.fit(X_train, Y_train)

y_pred_rf = forest.predict(X_test)
y_prob_rf = forest.predict_proba(X_test)[:, 1]
print(classification_report(Y_test, y_pred_rf, target_names=['rejected (0)', 'accepted (1)']))
print(f"ROC-AUC Score:          {roc_auc_score(Y_test, y_prob_rf):.4f}")
print(f"PR-AUC (Avg Precision): {average_precision_score(Y_test, y_prob_rf):.4f}")

              precision    recall  f1-score   support

rejected (0)       0.74      0.63      0.68      5698
accepted (1)       0.38      0.51      0.44      2540

    accuracy                           0.59      8238
   macro avg       0.56      0.57      0.56      8238
weighted avg       0.63      0.59      0.61      8238

ROC-AUC Score:          0.6241
PR-AUC (Avg Precision): 0.4132


In [60]:
from xgboost import XGBClassifier

pos_ratio = (Y_train == 0).sum() / (Y_train == 1).sum()

boost=Pipeline(
    steps=[
        ('preprocessor', rf_prep),
        ('classifier', XGBClassifier(
            n_estimators=100,
            scale_pos_weight=pos_ratio,
            random_state=42,
            n_jobs=-1,
            max_depth=5
        ))
    ]    
)

boost.fit(X_train, Y_train)
y_pred_xg = boost.predict(X_test)
y_prob_xg = boost.predict_proba(X_test)[:, 1]
print(classification_report(Y_test, y_pred_xg, target_names=['rejected (0)', 'accepted (1)']))
print(f"ROC-AUC Score:          {roc_auc_score(Y_test, y_prob_xg):.4f}")
print(f"PR-AUC (Avg Precision): {average_precision_score(Y_test, y_prob_xg):.4f}")

              precision    recall  f1-score   support

rejected (0)       0.71      0.85      0.77      5698
accepted (1)       0.39      0.22      0.28      2540

    accuracy                           0.66      8238
   macro avg       0.55      0.53      0.53      8238
weighted avg       0.61      0.66      0.62      8238

ROC-AUC Score:          0.5562
PR-AUC (Avg Precision): 0.3636


In [61]:
print("Columns in X_train:\n", list(X_train.columns))
print("\nTrain macro ranges:")
print("euribor3m:", X_train["euribor3m"].min(), "to", X_train["euribor3m"].max())
print("nr.employed:", X_train["nr.employed"].min(), "to", X_train["nr.employed"].max())

print("\nTest macro ranges:")
print("euribor3m:", X_test["euribor3m"].min(), "to", X_test["euribor3m"].max())
print("nr.employed:", X_test["nr.employed"].min(), "to", X_test["nr.employed"].max())

Columns in X_train:
 ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'pdays_cat']

Train macro ranges:
euribor3m: 1.299 to 5.045
nr.employed: 5099.1 to 5228.1

Test macro ranges:
euribor3m: 0.634 to 1.299
nr.employed: 4963.6 to 5099.1


In [62]:
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis,QuadraticDiscriminantAnalysis


lda_features = ["age", "campaign", "previous", "cons.price.idx", "cons.conf.idx"]

preprocessor_da = ColumnTransformer(
    transformers=[("scaler", StandardScaler(), lda_features)], remainder="drop"
)

lda_pipeline = Pipeline(
    steps=[
        ("prep", preprocessor_da),
        ("clf", LinearDiscriminantAnalysis(solver="svd")),
    ]
)

lda_pipeline.fit(X_train, Y_train)
lda_probs = lda_pipeline.predict_proba(X_test)[:, 1]

qda_pipeline = Pipeline(
    steps=[
        ("prep", preprocessor_da),
        ("clf", QuadraticDiscriminantAnalysis(reg_param=0.1)),
    ]
)

qda_pipeline.fit(X_train, Y_train)
qda_probs = qda_pipeline.predict_proba(X_test)[:, 1]

# Evaluate
print("=== LDA Performance ===")
print(f"ROC-AUC: {roc_auc_score(Y_test, lda_probs):.4f}")
print(f"PR-AUC:  {average_precision_score(Y_test, lda_probs):.4f}")

print("\n=== QDA Performance ===")
print(f"ROC-AUC: {roc_auc_score(Y_test, qda_probs):.4f}")
print(f"PR-AUC:  {average_precision_score(Y_test, qda_probs):.4f}")

=== LDA Performance ===
ROC-AUC: 0.2788
PR-AUC:  0.2142

=== QDA Performance ===
ROC-AUC: 0.5046
PR-AUC:  0.3858
